# Session 2: dimensional analysis
**A. Real-world problem.** Units can disagree without anyone noticing: an engineer adds a
length to a time, or feeds feet into a formula written for metres. Before trusting any
equation we can check its *dimensions*: every term must be the same kind of quantity.
Today you write each kind of quantity as a vector of exponents of mass (M), length (L) and
time (T), check equations with it, and measure how big a hidden unit mistake becomes.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
G = 9.80665          # standard gravity, m/s^2

## Step 1: a dimension is a vector of exponents
speed = length / time = L T^-1, so its vector is [M, L, T] = [0, 1, -1].
Multiplying quantities adds the vectors; dividing subtracts them; a power multiplies them.

In [ ]:
DIMS = {
    "dimensionless": [0, 0, 0], "mass": [1, 0, 0], "length": [0, 1, 0], "time": [0, 0, 1],
    "area": [0, 2, 0], "volume": [0, 3, 0], "speed": [0, 1, -1], "acceleration": [0, 1, -2],
    "force": [1, 1, -2], "impulse": [1, 1, -1], "momentum": [1, 1, -1], "energy": [1, 2, -2],
    "pressure": [1, -1, -2], "density": [1, -3, 0], "viscosity": [1, -1, -1],
}


def dim(x):
    """Dimension vector from a name (like 'force') or from a list [M, L, T]."""
    return np.array(DIMS[x] if isinstance(x, str) else x, dtype=float)


def dim_str(d):
    names = ("M", "L", "T")
    parts = [f"{n}^{e:g}" if e != 1 else n for n, e in zip(names, d) if e != 0]
    return " ".join(parts) if parts else "1 (no dimension)"


def expr_dim(factors):
    """Dimension of a product of factors [(quantity, power), ...]."""
    total = np.zeros(3)
    for q, power in factors:
        total = total + power * dim(q)
    return total


def same_dim(a, b):
    return bool(np.allclose(dim(a), dim(b)))

## Step 2: check an equation
An equation  left = term1 + term2  is dimensionally possible only if every term has the same
dimension as the left side. (Passing the test does not prove the equation is right; failing
it proves the equation is wrong.)

In [ ]:
def check_sum(left, terms):
    """left: a quantity; terms: list of factor lists. Returns one True/False per term."""
    return [same_dim(expr_dim(t), left) for t in terms]

## Step 3: let dimensions find the formula
A pendulum's period T might depend on its length L, gravity g and mass m:  T ~ L^p g^q m^r.
Match exponents of M, L, T on both sides. That is three linear equations, solved with numpy.

In [ ]:
def find_exponents(quantities, target):
    """Exponents e so that product(quantity_i ** e_i) has the dimension `target`."""
    A = np.array([dim(q) for q in quantities]).T       # columns = dimension vectors
    return np.linalg.solve(A, dim(target))

## Step 4: how big does a hidden unit mistake get?
If a length is wrong by a factor f (for example feet read as metres), a quantity with
dimension L^b is wrong by f^b. Mass factor m and time factor t work the same way:
    error factor = m_f^M * l_f^L * t_f^T.
For independent measurements with small relative errors, errors add up as shown below.

In [ ]:
def scale_factor(d, mass=1.0, length=1.0, time=1.0):
    """How much a quantity of dimension d is multiplied by when units are rescaled."""
    d = dim(d)
    return float(mass ** d[0] * length ** d[1] * time ** d[2])


def rel_error_worst(powers, rel_errors):
    """Worst case relative error of a product x1^n1 * x2^n2 ... (errors all push the same way)."""
    return float(np.sum(np.abs(powers) * np.abs(rel_errors)))


def rel_error_rss(powers, rel_errors):
    """Typical relative error of a product when the errors are independent (root sum of squares)."""
    return float(np.sqrt(np.sum((np.asarray(powers) * np.asarray(rel_errors)) ** 2)))


def pendulum_period(length_m, g=G):
    """Small-swing period of a simple pendulum, in seconds (2 pi comes from physics, not dimensions)."""
    return 2 * np.pi * np.sqrt(length_m / g)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    good = check_sum("length", [[("speed", 1), ("time", 1)], [("acceleration", 1), ("time", 2)]])
    bad = check_sum("length", [[("speed", 1), ("time", 1)], [("acceleration", 1), ("time", 1)]])
    print("s = u t + a t^2 (per term, ok?):", good, "  s = u t + a t:", bad)
    p, q, r = find_exponents(["length", "acceleration", "mass"], "time")
    print(f"pendulum period ~ L^{p:g} g^{q:g} m^{r:g}")
    print(f"period of a 1.000 m pendulum: {pendulum_period(1.0):.3f} s")
    for name in ("length", "area", "volume", "speed", "force"):
        print(f"feet read as metres: {name:7s} {dim_str(dim(name)):10s} wrong by x{1 / scale_factor(name, length=0.3048):.2f}")

    names = ["length", "area", "volume"]
    wrong = [1 / scale_factor(n, length=0.3048) for n in names]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(names, wrong, color=["#2a9d8f", "#e9c46a", "#d9534f"])
    for i, w in enumerate(wrong):
        ax.text(i, w + 0.5, f"x{w:.2f}", ha="center")
    ax.set_xlabel("quantity (dimension L^1, L^2, L^3)")
    ax.set_ylabel("error factor when feet are read as metres")
    ax.set_title("A unit mistake grows with the power of length")
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(OUT / "error_factors.png", dpi=120)

    length = np.linspace(0.05, 2, 100)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(length, pendulum_period(length), label="2 pi sqrt(L/g)  (correct dimensions)")
    ax2.plot(length, length / G * 2 * np.pi, "--", label="2 pi L/g  (wrong dimensions)")
    ax2.set_xlabel("pendulum length (m)")
    ax2.set_ylabel("computed value (s for the solid curve)")
    ax2.set_title("Only one power of L and g gives a time")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "pendulum_dimensions.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the README questions (F), then run the checkpoint.

## Coding tasks
1. **Easy.** Print `dim_str` for pressure, energy and density in `lesson.py`. Predict each one first.
2. **Stretch.** Use `find_exponents` to find how the speed v of a falling object depends on g and the height h. Confirm that v ~ sqrt(g h).
3. **Explain.** Explain why the same feet-for-metres mistake makes a length wrong by 3.28 but a volume wrong by 35.3.

## Assessment
- **A. Conceptual.** A formula passes the dimension check. Does that prove it is correct? Give an example and say what a failing check proves.
- **B. Numerical.** A tank volume of 12.0 'cubic metres' was really 12.0 cubic feet. By what factor is the volume overstated, and what is the true volume in litres?
- **C. Graph and simulation interpretation.** In the app with p = 1 and q = -1, what dimension does the formula 2 pi L^p g^q have, and why is the curve far from the grey period curve?
- **D. Design and optimisation challenge.** Design: a mass m on a spring of stiffness k (force per length) oscillates with period T. Use dimensions to find how T depends on m and k, then say what dimensional analysis cannot tell you.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    p, q, r = L.find_exponents(["length", "acceleration", "mass"], "time")
    checks.append(("worked example: pendulum period goes as L^0.5 g^-0.5 m^0",
                   np.allclose([p, q, r], [0.5, -0.5, 0.0], atol=1e-9)))
    checks.append(("worked example: a 1.000 m pendulum swings in 2.006 s",
                   abs(L.pendulum_period(1.0) - 2.006) < 5e-4))
    ok_terms = L.check_sum("length", [[("speed", 1), ("time", 1)], [("acceleration", 1), ("time", 2)]])
    checks.append(("s = u t + a t^2 is dimensionally consistent", all(ok_terms)))
    bad_terms = L.check_sum("length", [[("speed", 1), ("time", 1)], [("acceleration", 1), ("time", 1)]])
    checks.append(("s = u t + a t is caught (second term is a speed)", bad_terms == [True, False]))
    checks.append(("impulse and momentum have the same dimension", L.same_dim("impulse", "momentum")))
    checks.append(("force times time is an impulse",
                   L.same_dim(L.expr_dim([("force", 1), ("time", 1)]), "impulse")))
    checks.append(("hand calculation: feet read as metres makes a volume 35.31 times too big",
                   abs(1 / L.scale_factor("volume", length=0.3048) - 35.3147) < 1e-3))
    re_dim = L.expr_dim([("density", 1), ("speed", 1), ("length", 1), ("viscosity", -1)])
    checks.append(("Reynolds number rho v D / mu has no dimension", bool(np.allclose(re_dim, 0.0))))
    checks.append(("worst-case error of x*y^2 with 2% and 1% errors is 4%",
                   abs(L.rel_error_worst([1, 2], [0.02, 0.01]) - 0.04) < 1e-12))
    checks.append(("independent errors (root sum of squares) are smaller: 2.83%",
                   abs(L.rel_error_rss([1, 2], [0.02, 0.01]) - 0.028284) < 1e-5))
    try:
        L.find_exponents(["length", "length", "mass"], "time")
        edge_ok = False
    except np.linalg.LinAlgError:
        edge_ok = True
    checks.append(("edge: dependent quantities cannot fix exponents (error raised)", edge_ok))
    checks.append(("no change of units, no change of value", L.scale_factor("force") == 1.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
